# Netflix Data Analytics + AI Project
### IBM SkillsBuild Data Analytics with AI – Academic Internship

**Project:** Netflix Movies and TV Shows – Exploratory Data Analysis and AI-Based Content Recommendation  
**Student:** Arpitakhan  
**Tools:** Python, Pandas, NumPy, Matplotlib, Scikit-learn, Jupyter Notebook

---

## 1. Project Overview

This project analyzes the `netflix_titles.csv` dataset to understand the composition and characteristics of Netflix movies and TV shows. The analysis covers content type, release year, countries, ratings, genres, movie duration, TV-show seasons, and missing data.

The project also includes a beginner-friendly **AI/ML component**: a content-based recommendation system. It uses **TF-IDF** text features and **cosine similarity** to recommend titles with similar descriptions and genres.

> **Important:** Keep `netflix_titles.csv` in the same folder as this notebook before running the notebook.


## 2. Objectives

1. Load and inspect the Netflix dataset.
2. Clean and prepare important columns.
3. Analyze missing values and data quality.
4. Compare Movies and TV Shows.
5. Identify common ratings, genres, and countries.
6. Study content release and Netflix-added trends.
7. Analyze movie duration and TV-show seasons.
8. Create clear visualizations for important patterns.
9. Build a simple AI-based content recommendation system.
10. Summarize findings, limitations, and possible future improvements.


## 3. Dataset

The commonly distributed Netflix Titles dataset contains fields such as:

- `show_id` – unique title identifier
- `type` – Movie or TV Show
- `title` – title name
- `director` – director(s)
- `cast` – cast members
- `country` – production country/countries
- `date_added` – date added to Netflix
- `release_year` – original release year
- `rating` – content rating
- `duration` – movie minutes or TV-show seasons
- `listed_in` – genre/category information
- `description` – short title description

**Dataset source:** Kaggle – Netflix Movies and TV Shows  
https://www.kaggle.com/datasets/shivamb/netflix-shows

The notebook is intentionally written to work with the local `netflix_titles.csv` file rather than relying on an internet connection during execution.


In [ ]:
# 4. Import libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print("Libraries imported successfully.")


In [ ]:
# 5. Load the dataset
FILE_NAME = "netflix_titles.csv"

try:
    df = pd.read_csv(FILE_NAME)
except FileNotFoundError:
    raise FileNotFoundError(
        "netflix_titles.csv was not found. Place the CSV file in the same folder as this notebook."
    )

print(f"Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()


In [ ]:
# 6. Check structure and data types
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())


In [ ]:
# 7. Basic dataset summary
print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Unique titles:", df["title"].nunique())

print("\nContent type counts:")
display(df["type"].value_counts())

print("\nNumerical summary:")
display(df.describe(include="all").T)


## 4. Data Quality and Missing Values

The dataset contains missing values, especially in fields such as director, cast, and country. Instead of deleting a large number of rows, this project keeps the records and uses an `Unknown` label where appropriate for categorical/text analysis.


In [ ]:
# 8. Missing-value analysis
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_report = pd.DataFrame({
    "Missing Values": missing,
    "Missing Percentage": missing_pct.round(2)
})

display(missing_report)


In [ ]:
# 9. Remove exact duplicate rows and standardize text fields
df = df.drop_duplicates().copy()

text_columns = [
    "type", "title", "director", "cast", "country",
    "rating", "duration", "listed_in", "description"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown").astype(str).str.strip()

# Convert dates safely
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")

# Convert release year safely
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce").astype("Int64")

print("Cleaning completed.")
print("New shape:", df.shape)


## 5. Feature Engineering

Additional fields are created to make the analysis easier:

- `added_year` – year the title was added to Netflix
- `added_month` – month the title was added
- `movie_duration_min` – numeric movie duration
- `tv_seasons` – numeric number of seasons
- `release_to_added_years` – approximate gap between release and Netflix addition


In [ ]:
# 10. Create useful analytical features
df["added_year"] = df["date_added"].dt.year
df["added_month"] = df["date_added"].dt.month

df["movie_duration_min"] = pd.to_numeric(
    df["duration"].str.extract(r"(\d+)")[0],
    errors="coerce"
)

df["tv_seasons"] = pd.to_numeric(
    df["duration"].str.extract(r"(\d+)")[0],
    errors="coerce"
)

df["release_to_added_years"] = (
    df["added_year"] - df["release_year"].astype("float")
)

display(df[[
    "title", "type", "release_year", "date_added",
    "duration", "movie_duration_min", "tv_seasons",
    "release_to_added_years"
]].head(10))


# 6. Exploratory Data Analysis (EDA)

The following sections answer practical questions about the Netflix catalog using tables and visualizations.


In [ ]:
# 11. Movies vs TV Shows
type_counts = df["type"].value_counts()

plt.figure(figsize=(7, 5))
type_counts.plot(kind="bar")
plt.title("Netflix Content by Type")
plt.xlabel("Content Type")
plt.ylabel("Number of Titles")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(type_counts.to_frame("Title Count"))


In [ ]:
# 12. Titles added to Netflix by year
added_by_year = df["added_year"].dropna().astype(int).value_counts().sort_index()

plt.figure(figsize=(10, 5))
added_by_year.plot(kind="line", marker="o")
plt.title("Titles Added to Netflix by Year")
plt.xlabel("Year Added")
plt.ylabel("Number of Titles")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

display(added_by_year.tail(15).to_frame("Titles Added"))


In [ ]:
# 13. Release year distribution
release_counts = df["release_year"].dropna().astype(int).value_counts().sort_index()

plt.figure(figsize=(12, 5))
release_counts.plot(kind="line")
plt.title("Netflix Titles by Original Release Year")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.tight_layout()
plt.show()


In [ ]:
# 14. Top production countries
country_series = (
    df["country"]
    .replace("Unknown", np.nan)
    .dropna()
    .str.split(", ")
    .explode()
    .str.strip()
)

top_countries = country_series.value_counts().head(10)

plt.figure(figsize=(9, 5))
top_countries.sort_values().plot(kind="barh")
plt.title("Top 10 Production Countries")
plt.xlabel("Number of Title-Country Records")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

display(top_countries.to_frame("Count"))


In [ ]:
# 15. Top genres/categories
genre_series = (
    df["listed_in"]
    .replace("Unknown", np.nan)
    .dropna()
    .str.split(", ")
    .explode()
    .str.strip()
)

top_genres = genre_series.value_counts().head(15)

plt.figure(figsize=(10, 6))
top_genres.sort_values().plot(kind="barh")
plt.title("Top 15 Netflix Genres / Categories")
plt.xlabel("Number of Titles")
plt.ylabel("Genre / Category")
plt.tight_layout()
plt.show()

display(top_genres.to_frame("Count"))


In [ ]:
# 16. Ratings distribution
rating_counts = df["rating"].value_counts().head(15)

plt.figure(figsize=(9, 5))
rating_counts.sort_values().plot(kind="barh")
plt.title("Most Common Content Ratings")
plt.xlabel("Number of Titles")
plt.ylabel("Rating")
plt.tight_layout()
plt.show()

display(rating_counts.to_frame("Title Count"))


In [ ]:
# 17. Movie duration analysis
movies = df[df["type"].str.lower() == "movie"].copy()

movie_duration = movies["movie_duration_min"].dropna()

print("Number of movies with numeric duration:", len(movie_duration))
print("Average movie duration:", round(movie_duration.mean(), 2), "minutes")
print("Median movie duration:", round(movie_duration.median(), 2), "minutes")

plt.figure(figsize=(9, 5))
plt.hist(movie_duration, bins=30)
plt.title("Distribution of Movie Durations")
plt.xlabel("Duration (minutes)")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()


In [ ]:
# 18. TV Show season analysis
tv_shows = df[df["type"].str.lower() == "tv show"].copy()
tv_seasons = tv_shows["tv_seasons"].dropna()

print("Number of TV shows with numeric season information:", len(tv_seasons))
print("Average seasons:", round(tv_seasons.mean(), 2))
print("Median seasons:", round(tv_seasons.median(), 2))

season_counts = tv_seasons.astype(int).value_counts().sort_index().head(15)

plt.figure(figsize=(9, 5))
season_counts.plot(kind="bar")
plt.title("TV Shows by Number of Seasons")
plt.xlabel("Number of Seasons")
plt.ylabel("Number of TV Shows")
plt.tight_layout()
plt.show()


In [ ]:
# 19. Content released in recent years
recent = (
    df.dropna(subset=["release_year"])
      .groupby(["release_year", "type"])
      .size()
      .unstack(fill_value=0)
      .tail(15)
)

display(recent)

recent.plot(kind="bar", figsize=(11, 5))
plt.title("Recent Titles by Type")
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Data-Driven Questions

The next cells produce concise answers that can be used in the project report.

### Questions
- How many Movies and TV Shows are present?
- Which content type is more common?
- Which countries appear most often?
- Which genres/categories are most common?
- What are the most common ratings?
- What is the typical movie duration?
- How many seasons do TV shows typically have?


In [ ]:
# 20. Automated key metrics
metrics = {
    "Total titles": len(df),
    "Movies": int((df["type"].str.lower() == "movie").sum()),
    "TV Shows": int((df["type"].str.lower() == "tv show").sum()),
    "Unique titles": int(df["title"].nunique()),
    "Earliest release year": int(df["release_year"].min()) if df["release_year"].notna().any() else None,
    "Latest release year": int(df["release_year"].max()) if df["release_year"].notna().any() else None,
}

for key, value in metrics.items():
    print(f"{key}: {value}")

if not top_countries.empty:
    print("\nTop production country:", top_countries.index[0])

if not top_genres.empty:
    print("Top genre/category:", top_genres.index[0])

if not rating_counts.empty:
    print("Most common rating:", rating_counts.index[0])

if not movie_duration.empty:
    print("Average movie duration (minutes):", round(movie_duration.mean(), 2))

if not tv_seasons.empty:
    print("Average TV-show seasons:", round(tv_seasons.mean(), 2))


# 8. AI Component – Content-Based Recommendation System

This section adds an AI/ML element without requiring a paid API.

### Method
1. Combine the title's `listed_in` categories and `description`.
2. Convert the text into numerical **TF-IDF** vectors.
3. Calculate **cosine similarity** between titles.
4. For a selected title, return the most similar titles.

**Why this is useful:** It demonstrates a practical recommendation workflow using machine learning concepts while remaining easy to understand and reproduce.


In [ ]:
# 21. Prepare text features for the recommender
recommend_df = df[["title", "type", "listed_in", "description"]].copy()

recommend_df["combined_text"] = (
    recommend_df["listed_in"].fillna("Unknown") + " " +
    recommend_df["description"].fillna("Unknown")
)

recommend_df = recommend_df.drop_duplicates(subset=["title"]).reset_index(drop=True)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(recommend_df["combined_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)


In [ ]:
# 22. Recommendation function
def recommend_titles(title, n=5):
    """Return titles similar to the supplied title using TF-IDF + cosine similarity."""
    matches = recommend_df[
        recommend_df["title"].str.lower() == str(title).strip().lower()
    ]

    if matches.empty:
        partial = recommend_df[
            recommend_df["title"].str.contains(
                str(title).strip(), case=False, na=False
            )
        ]
        if partial.empty:
            return f"No title found for: {title}"
        idx = partial.index[0]
        selected_title = partial.iloc[0]["title"]
        print(f"Using closest title match: {selected_title}")
    else:
        idx = matches.index[0]
        selected_title = matches.iloc[0]["title"]

    similarity_scores = cosine_similarity(
        tfidf_matrix[idx],
        tfidf_matrix
    ).flatten()

    similar_indices = similarity_scores.argsort()[::-1]

    results = []
    for i in similar_indices:
        if i == idx:
            continue
        results.append({
            "Title": recommend_df.loc[i, "title"],
            "Type": recommend_df.loc[i, "type"],
            "Similarity": round(float(similarity_scores[i]), 3)
        })
        if len(results) >= n:
            break

    return pd.DataFrame(results)

# Example: use the first title in the dataset so the notebook works
example_title = recommend_df.iloc[0]["title"]
print("Example title:", example_title)
display(recommend_titles(example_title, n=5))


### How the AI recommendation works

**TF-IDF** gives higher importance to words that are informative for a title and lower importance to words that appear in many documents.

**Cosine similarity** measures how close two TF-IDF vectors are. A higher similarity value means the two title descriptions/categories have more similar text.

This is a **content-based** recommender. It does not use personal watch history, ratings, or user accounts.


In [ ]:
# 23. Try your own recommendation
# Replace the title below with any exact or partial title from the dataset.
user_title = example_title

recommendations = recommend_titles(user_title, n=10)
display(recommendations)


# 9. Key Findings Template

After running all cells, record the values printed by the notebook in the final report. Typical findings from this dataset include:

- The catalog contains both Movies and TV Shows, with Movies forming the larger share in the standard 8,807-row version.
- Production countries and genres are multi-valued fields, so one title can contribute to more than one country/genre count.
- Many rows have missing director, cast, and country information; these should not automatically be treated as zero or as evidence that the title has no director/cast.
- Movie duration is stored as text such as `90 min`, while TV-show duration is stored as values such as `2 Seasons`; separating these fields is necessary for correct numeric analysis.
- The recommendation model uses textual similarity, so a recommendation means “similar content description/category,” not “guaranteed user preference.”


# 10. Limitations

1. The dataset represents a particular snapshot and should not be interpreted as the current Netflix catalog.
2. Missing values reduce the completeness of country, director, and cast analysis.
3. Multiple countries and genres are stored in one cell, so counts represent title-country or title-genre occurrences rather than mutually exclusive titles.
4. The recommendation model uses only available text fields; it does not learn from actual user behavior.
5. TF-IDF recommendations depend on wording and may miss semantic similarities that a modern language model could detect.
6. The project does not claim that Netflix itself uses this exact recommendation algorithm.


# 11. Conclusion

This project demonstrates an end-to-end beginner-friendly data analytics workflow:

**Load → Inspect → Clean → Transform → Analyze → Visualize → Apply AI → Interpret → Document**

The EDA provides an overview of Netflix content characteristics, while the AI section demonstrates how text data can be transformed into numerical features and used for content-based recommendations.

The notebook is designed to be reproducible: place the supplied `netflix_titles.csv` file beside the notebook and run the cells from top to bottom.


## 12. Suggested Future Improvements

- Build an interactive dashboard using Power BI or Tableau.
- Add sentiment analysis on descriptions.
- Use word embeddings or transformer models for semantic recommendations.
- Add a user-profile-based recommender using collaborative filtering.
- Compare recommendation quality using evaluation metrics.
- Build a small Streamlit application around the recommender.


## 13. References

1. Kaggle – Netflix Movies and TV Shows dataset: https://www.kaggle.com/datasets/shivamb/netflix-shows
2. Pandas documentation: https://pandas.pydata.org/docs/
3. Scikit-learn documentation: https://scikit-learn.org/stable/
4. Matplotlib documentation: https://matplotlib.org/stable/

**Project note:** This project uses the local CSV supplied for submission. The external dataset link is included for dataset attribution and reproducibility.
